# DNS Do53 / DoT Migration Test Report

This notebook validates recursive DNS server behavior over **Do53** (plain UDP/TCP port 53) and **DNS-over-TLS** (DoT, port 853), then benchmarks performance under increasing DoT load.

**Sections:**
1. [Configuration](#1.-Configuration)
2. [Helper Functions](#2.-Helper-Functions)
3. [Do53 Functional Tests](#3.-Do53-Functional-Tests)
4. [DoT Functional Tests](#4.-DoT-Functional-Tests)
5. [Do53 Performance Baseline](#5.-Do53-Performance-Baseline)
6. [DoT Performance Baseline](#6.-DoT-Performance-Baseline)
7. [Do53 vs DoT Comparison](#7.-Do53-vs-DoT-Comparison)
8. [Scaling Test: Mixed Do53/DoT Traffic](#8.-Scaling-Test:-Mixed-Do53/DoT-Traffic)
9. [Executive Summary](#9.-Executive-Summary)
10. [Export](#10.-Export)

## 1. Configuration

Edit the variables below to match your test environment.

In [ ]:
# ── Servers under test ──────────────────────────────────────────────
# List of recursive DNS servers to test (hostnames or IPs).
# Hostnames are recommended for DoT so TLS certificate CN validation works.
DNS_SERVERS = ["1.1.1.1"]

# ── TLS settings ────────────────────────────────────────────────────
# Path to a custom CA certificate bundle for DoT verification.
# Set to "" or None to use the system default CA store.
TLS_CA_FILE = ""  # e.g. "/workspace/certs/my-ca.pem"

# ── Queries for functional tests ────────────────────────────────────
TEST_QUERIES = [
    {"name": "example.com",       "qtype": "A"},
    {"name": "example.com",       "qtype": "AAAA"},
    {"name": "example.com",       "qtype": "MX"},
    {"name": "example.com",       "qtype": "SOA"},
    {"name": "www.example.com",   "qtype": "A"},
    {"name": "cloudflare.com",    "qtype": "A"},
    {"name": "google.com",        "qtype": "A"},
]

# ── Query input file for performance tests ──────────────────────────
# dnsperf format: one "domain qtype" per line
# If this file does not exist it will be auto-generated from TEST_QUERIES
QUERY_INPUT_FILE = "/tmp/perf-queries.txt"

# ── Performance parameters ──────────────────────────────────────────
PERF_DURATION_SEC = 30       # Duration of each performance run
PERF_TOTAL_QPS    = 500      # Target queries-per-second (total)
PERF_CONCURRENCY  = 10       # Concurrent workers

# ── Scaling test steps ──────────────────────────────────────────────
# Each value is the DoT percentage of total QPS (remainder is Do53)
SCALING_STEPS = [10, 20, 30, 40, 50, 60, 70, 80, 90]
SCALING_DURATION_SEC = 15    # Duration per scaling step

# ── Timeouts ────────────────────────────────────────────────────────
TOOL_TIMEOUT_SEC = 120

# ── Resolve server hostnames to IPs (needed for dnspython) ──────────
import os, socket

def resolve_to_ip(server):
    """Resolve a hostname to an IP address. Returns as-is if already an IP."""
    try:
        socket.inet_aton(server)
        return server  # already an IP
    except OSError:
        pass
    try:
        socket.inet_pton(socket.AF_INET6, server)
        return server  # already an IPv6
    except OSError:
        pass
    # It's a hostname — resolve it
    return socket.getaddrinfo(server, None, socket.AF_INET)[0][4][0]

SERVER_IPS = {}
for _srv in DNS_SERVERS:
    _ip = resolve_to_ip(_srv)
    SERVER_IPS[_srv] = _ip
    if _ip != _srv:
        print(f"Resolved {_srv} -> {_ip}")
    else:
        print(f"Server: {_srv} (IP)")

# ── Auto-generate query file if missing ─────────────────────────────
if not os.path.exists(QUERY_INPUT_FILE):
    with open(QUERY_INPUT_FILE, "w") as f:
        for q in TEST_QUERIES:
            f.write(f"{q['name']} {q['qtype']}\n")
    print(f"Generated {QUERY_INPUT_FILE} with {len(TEST_QUERIES)} queries")
else:
    with open(QUERY_INPUT_FILE) as f:
        count = sum(1 for line in f if line.strip())
    print(f"Using existing {QUERY_INPUT_FILE} ({count} queries)")

# Validate CA file if specified
if TLS_CA_FILE:
    if os.path.exists(TLS_CA_FILE):
        print(f"Custom CA: {TLS_CA_FILE}")
    else:
        print(f"WARNING: TLS_CA_FILE not found: {TLS_CA_FILE}")
else:
    print("TLS CA: system default")

print(f"Performance: {PERF_TOTAL_QPS} QPS target, {PERF_DURATION_SEC}s duration")
print(f"Scaling steps: {SCALING_STEPS}")

## 2. Helper Functions

In [ ]:
import subprocess, json, re, io, time
import concurrent.futures
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from rich.console import Console
from rich.table import Table
from rich.text import Text
from IPython.display import display, HTML

console = Console(force_jupyter=True, width=120)

# ── Chart styling ───────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
})
COLOR_DO53  = "#2196F3"   # blue
COLOR_DOT   = "#FF9800"   # orange
COLOR_ERROR = "#F44336"   # red
COLOR_OK    = "#4CAF50"   # green

# ── Subprocess runner ───────────────────────────────────────────────
def run_cmd(cmd, timeout=TOOL_TIMEOUT_SEC):
    """Run a command and return (stdout, stderr, returncode)."""
    try:
        proc = subprocess.run(
            cmd, capture_output=True, text=True, timeout=timeout
        )
        return proc.stdout, proc.stderr, proc.returncode
    except subprocess.TimeoutExpired:
        return "", f"TIMEOUT after {timeout}s", -1
    except Exception as e:
        return "", str(e), -1

# ── dnsperf output parser ───────────────────────────────────────────
def parse_dnsperf(output):
    """Parse dnsperf text output into a results dict."""
    result = {}
    patterns = {
        "queries_sent":     r"Queries sent:\s+(\d+)",
        "queries_completed": r"Queries completed:\s+(\d+)",
        "queries_lost":     r"Queries lost:\s+(\d+)",
        "qps":              r"Queries per second:\s+([\d.]+)",
        "latency_avg_s":    r"Average Latency \(s\):\s+([\d.]+)",
        "latency_min_s":    r"Minimum Latency \(s\):\s+([\d.]+)",
        "latency_max_s":    r"Maximum Latency \(s\):\s+([\d.]+)",
        "run_time_s":       r"Run time \(s\):\s+([\d.]+)",
    }
    for key, pattern in patterns.items():
        m = re.search(pattern, output)
        if m:
            result[key] = float(m.group(1))
    # Convert latencies to ms
    for k in ["latency_avg_s", "latency_min_s", "latency_max_s"]:
        if k in result:
            result[k.replace("_s", "_ms")] = result.pop(k) * 1000
    return result

# ── dnspyre JSON parser ─────────────────────────────────────────────
def parse_dnspyre(output):
    """Parse dnspyre --json output into a results dict."""
    try:
        data = json.loads(output)
    except json.JSONDecodeError:
        return {}
    # dnspyre returns a list; take the first (and usually only) entry
    if isinstance(data, list) and data:
        data = data[0]
    latency = data.get("latencyStats", {})
    success = data.get("totalSuccessResponses", 0)
    failed  = data.get("totalFailedResponses", 0)
    total   = success + failed
    return {
        "qps":            data.get("queriesPerSecond", 0),
        "latency_avg_ms": latency.get("avgMs", 0),
        "latency_min_ms": latency.get("minMs", 0),
        "latency_max_ms": latency.get("maxMs", 0),
        "latency_p50_ms": latency.get("p50Ms", 0),
        "latency_p95_ms": latency.get("p95Ms", 0),
        "latency_p99_ms": latency.get("p99Ms", 0),
        "queries_completed": total,
        "queries_ok":     success,
        "queries_error":  failed,
    }

# ── Status helpers ──────────────────────────────────────────────────
def status_text(ok):
    return Text("PASS", style="bold green") if ok else Text("FAIL", style="bold red")

print("Helpers loaded.")

---
## 3. Do53 Functional Tests

Verify that each server correctly resolves queries over plain DNS (UDP port 53).

In [ ]:
import dns.resolver

do53_func_results = []

for server in DNS_SERVERS:
    server_ip = SERVER_IPS[server]

    for q in TEST_QUERIES:
        name, qtype = q["name"], q["qtype"]

        # Test with dig (uses hostname directly)
        stdout, stderr, rc = run_cmd(
            ["dig", f"@{server}", name, qtype, "+short", "+timeout=5", "+tries=2"],
            timeout=15
        )
        dig_answer = stdout.strip()
        dig_ok = rc == 0 and len(dig_answer) > 0

        # Cross-validate with dnspython (requires IP address)
        try:
            resolver = dns.resolver.Resolver()
            resolver.nameservers = [server_ip]
            resolver.lifetime = 5
            answers = resolver.resolve(name, qtype)
            py_answer = ", ".join(str(a) for a in answers)
            py_ok = True
        except Exception as e:
            py_answer = str(e)
            py_ok = False

        do53_func_results.append({
            "server": server, "name": name, "qtype": qtype,
            "dig_ok": dig_ok, "dig_answer": dig_answer[:60],
            "py_ok": py_ok, "py_answer": py_answer[:60],
            "pass": dig_ok and py_ok,
        })

# DNSSEC check (kdig uses hostname directly)
for server in DNS_SERVERS:
    stdout, stderr, rc = run_cmd(
        ["kdig", f"@{server}", "example.com", "A", "+dnssec", "+timeout=5"],
        timeout=15
    )
    ad_set = bool(re.search(r"flags:.*\bAD\b", stdout))
    do53_func_results.append({
        "server": server, "name": "example.com", "qtype": "DNSSEC",
        "dig_ok": ad_set, "dig_answer": "AD flag set" if ad_set else "AD flag missing",
        "py_ok": ad_set, "py_answer": "-",
        "pass": ad_set,
    })

In [ ]:
table = Table(title="Do53 Functional Test Results", show_lines=True)
table.add_column("Server", style="cyan")
table.add_column("Query")
table.add_column("Type")
table.add_column("Status", justify="center")
table.add_column("Answer (dig)")
table.add_column("Answer (dnspython)")

for r in do53_func_results:
    table.add_row(
        r["server"], r["name"], r["qtype"],
        status_text(r["pass"]),
        r["dig_answer"], r["py_answer"],
    )

passed = sum(1 for r in do53_func_results if r["pass"])
total = len(do53_func_results)
console.print(table)
console.print(f"\n[bold]{passed}/{total} tests passed[/bold]")

---
## 4. DoT Functional Tests

Verify TLS certificate validity and correct resolution over DNS-over-TLS (port 853).

In [ ]:
# TLS certificate inspection
cert_results = []
for server in DNS_SERVERS:
    stdout, stderr, rc = run_cmd(
        ["dot-cert-tester", server], timeout=30
    )
    output = stdout + stderr
    cert_ok = rc == 0 and "error" not in output.lower()
    cert_results.append({"server": server, "ok": cert_ok, "output": output.strip()})

table = Table(title="DoT TLS Certificate Check", show_lines=True)
table.add_column("Server", style="cyan")
table.add_column("Status", justify="center")
table.add_column("Details", max_width=90)

for r in cert_results:
    # Show first few lines of cert output
    details = "\n".join(r["output"].splitlines()[:8])
    table.add_row(r["server"], status_text(r["ok"]), details)

console.print(table)

In [ ]:
dot_func_results = []

# Build kdig TLS flag: +tls-ca=FILE if custom CA, else +tls-ca (system default)
kdig_tls_flag = f"+tls-ca={TLS_CA_FILE}" if TLS_CA_FILE else "+tls-ca"

# Build q TLS args
q_tls_args = ["--tls-ca=" + TLS_CA_FILE] if TLS_CA_FILE else []

for server in DNS_SERVERS:
    for q_entry in TEST_QUERIES:
        name, qtype = q_entry["name"], q_entry["qtype"]

        # Test with kdig over TLS
        stdout, stderr, rc = run_cmd(
            ["kdig", f"@{server}", name, qtype, kdig_tls_flag, "+short", "+timeout=5"],
            timeout=15
        )
        kdig_answer = stdout.strip()
        kdig_ok = rc == 0 and len(kdig_answer) > 0

        # Cross-validate with q over DoT
        q_cmd = ["q", name, qtype, f"@dot://{server}"] + q_tls_args
        stdout2, stderr2, rc2 = run_cmd(q_cmd, timeout=15)
        q_ok = rc2 == 0 and len(stdout2.strip()) > 0
        q_answer = stdout2.strip().splitlines()[0][:60] if stdout2.strip() else stderr2.strip()[:60]

        dot_func_results.append({
            "server": server, "name": name, "qtype": qtype,
            "kdig_ok": kdig_ok, "kdig_answer": kdig_answer[:60],
            "q_ok": q_ok, "q_answer": q_answer,
            "pass": kdig_ok and q_ok,
        })

In [ ]:
table = Table(title="DoT Functional Test Results", show_lines=True)
table.add_column("Server", style="cyan")
table.add_column("Query")
table.add_column("Type")
table.add_column("Status", justify="center")
table.add_column("Answer (kdig +tls-ca)")
table.add_column("Answer (q @dot://)")

for r in dot_func_results:
    table.add_row(
        r["server"], r["name"], r["qtype"],
        status_text(r["pass"]),
        r["kdig_answer"], r["q_answer"],
    )

passed = sum(1 for r in dot_func_results if r["pass"])
total = len(dot_func_results)
console.print(table)
console.print(f"\n[bold]{passed}/{total} tests passed[/bold]")

---
## 5. Do53 Performance Baseline

Sustained load test using **dnsperf** over plain DNS.

In [ ]:
do53_perf = []

for server in DNS_SERVERS:
    console.print(f"[bold]Running Do53 baseline on {server}...[/bold]")
    stdout, stderr, rc = run_cmd([
        "dnsperf",
        "-s", server,
        "-d", QUERY_INPUT_FILE,
        "-l", str(PERF_DURATION_SEC),
        "-Q", str(PERF_TOTAL_QPS),
        "-c", str(PERF_CONCURRENCY),
    ])
    parsed = parse_dnsperf(stdout + stderr)
    parsed["server"] = server
    parsed["protocol"] = "Do53"
    do53_perf.append(parsed)
    console.print(f"  QPS: {parsed.get('qps', 0):.1f}  "
                  f"Avg latency: {parsed.get('latency_avg_ms', 0):.2f} ms  "
                  f"Lost: {int(parsed.get('queries_lost', 0) or 0)}")

do53_perf_df = pd.DataFrame(do53_perf)
do53_perf_df

---
## 6. DoT Performance Baseline

Sustained load test using **dnspyre** over DNS-over-TLS.

In [ ]:
dot_perf = []

# Read query names from input file for dnspyre
with open(QUERY_INPUT_FILE) as f:
    query_domains = [line.strip().split()[0] for line in f if line.strip()]

# dnspyre CA args
dnspyre_tls_args = ["--tls-ca=" + TLS_CA_FILE] if TLS_CA_FILE else []

for server in DNS_SERVERS:
    console.print(f"[bold]Running DoT baseline on {server}...[/bold]")
    total_queries = PERF_TOTAL_QPS * PERF_DURATION_SEC
    cmd = [
        "dnspyre",
        "--server", f"{server}:853",
        "--tls",
        "--json",
        "-n", str(total_queries),
        "-c", str(PERF_CONCURRENCY),
        "--duration", f"{PERF_DURATION_SEC}s",
    ] + dnspyre_tls_args + query_domains
    stdout, stderr, rc = run_cmd(cmd)
    parsed = parse_dnspyre(stdout)
    parsed["server"] = server
    parsed["protocol"] = "DoT"
    dot_perf.append(parsed)
    console.print(f"  QPS: {parsed.get('qps', 0):.1f}  "
                  f"Avg latency: {parsed.get('latency_avg_ms', 0):.2f} ms  "
                  f"p95: {parsed.get('latency_p95_ms', 0):.2f} ms  "
                  f"Errors: {int(parsed.get('queries_error', 0) or 0)}")

dot_perf_df = pd.DataFrame(dot_perf)
dot_perf_df

---
## 7. Do53 vs DoT Comparison

Side-by-side comparison of baseline performance.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, server in enumerate(DNS_SERVERS):
    d53 = next((r for r in do53_perf if r["server"] == server), {})
    dot = next((r for r in dot_perf if r["server"] == server), {})

    # ── Latency comparison ──
    ax = axes[0]
    labels = ["Avg", "Min", "Max"]
    do53_vals = [d53.get("latency_avg_ms", 0), d53.get("latency_min_ms", 0), d53.get("latency_max_ms", 0)]
    dot_vals  = [dot.get("latency_avg_ms", 0), dot.get("latency_min_ms", 0), dot.get("latency_max_ms", 0)]
    x = np.arange(len(labels))
    w = 0.35
    ax.bar(x - w/2, do53_vals, w, label="Do53", color=COLOR_DO53)
    ax.bar(x + w/2, dot_vals, w, label="DoT", color=COLOR_DOT)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Latency (ms)")
    ax.set_title(f"Latency — {server}")
    ax.legend()

    # ── QPS comparison ──
    ax = axes[1]
    ax.bar(["Do53", "DoT"], [d53.get("qps", 0), dot.get("qps", 0)],
           color=[COLOR_DO53, COLOR_DOT])
    ax.set_ylabel("Queries / sec")
    ax.set_title(f"Throughput — {server}")

    # ── Error comparison ──
    ax = axes[2]
    do53_err = d53.get("queries_lost", 0) or 0
    dot_err  = dot.get("queries_error", 0) or 0
    ax.bar(["Do53", "DoT"], [do53_err, dot_err],
           color=[COLOR_DO53, COLOR_DOT])
    ax.set_ylabel("Failed queries")
    ax.set_title(f"Errors — {server}")

plt.suptitle("Do53 vs DoT Baseline Comparison", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Summary table
table = Table(title="Baseline Performance Summary", show_lines=True)
table.add_column("Server", style="cyan")
table.add_column("Protocol")
table.add_column("QPS", justify="right")
table.add_column("Avg (ms)", justify="right")
table.add_column("Min (ms)", justify="right")
table.add_column("Max (ms)", justify="right")
table.add_column("Errors", justify="right")

for r in do53_perf:
    table.add_row(r["server"], "Do53",
        f"{r.get('qps',0):.1f}", f"{r.get('latency_avg_ms',0):.2f}",
        f"{r.get('latency_min_ms',0):.2f}", f"{r.get('latency_max_ms',0):.2f}",
        str(int(r.get('queries_lost', 0) or 0)))
for r in dot_perf:
    table.add_row(r["server"], "DoT",
        f"{r.get('qps',0):.1f}", f"{r.get('latency_avg_ms',0):.2f}",
        f"{r.get('latency_min_ms',0):.2f}", f"{r.get('latency_max_ms',0):.2f}",
        str(int(r.get('queries_error', 0) or 0)))

console.print(table)

# Overhead calculation
for server in DNS_SERVERS:
    d53 = next((r for r in do53_perf if r["server"] == server), {})
    dot = next((r for r in dot_perf if r["server"] == server), {})
    avg53 = d53.get("latency_avg_ms", 0)
    avgdot = dot.get("latency_avg_ms", 0)
    if avg53 > 0:
        overhead = ((avgdot - avg53) / avg53) * 100
        console.print(f"\n[bold]{server}:[/bold] DoT latency overhead vs Do53: "
                      f"[{'red' if overhead > 50 else 'yellow' if overhead > 20 else 'green'}]"
                      f"{overhead:+.1f}%[/]")

---
## 8. Scaling Test: Mixed Do53/DoT Traffic

This test holds **total QPS constant** and gradually shifts traffic from Do53 to DoT.

| Step | Do53 QPS | DoT QPS |
|------|----------|---------|
| 10%  | 90%      | 10%     |
| 20%  | 80%      | 20%     |
| ...  | ...      | ...     |
| 90%  | 10%      | 90%     |

At each step, **dnsperf** (Do53) and **dnspyre** (DoT) run concurrently.
We record the actual achieved QPS and latency from each tool.

> **Note:** dnspyre does not have a strict QPS rate limiter; its actual throughput
> is governed by concurrency and server response time. The charts below show
> *measured* values, not targets.

In [ ]:
scaling_results = []

with open(QUERY_INPUT_FILE) as f:
    query_domains = [line.strip().split()[0] for line in f if line.strip()]

# dnspyre CA args for scaling
dnspyre_tls_args = ["--tls-ca=" + TLS_CA_FILE] if TLS_CA_FILE else []

for server in DNS_SERVERS:
    console.print(f"\n[bold cyan]Scaling test for {server}[/bold cyan]")
    console.print(f"{'Step':>6}  {'Do53 QPS':>10}  {'DoT QPS':>10}  "
                  f"{'Do53 Avg':>10}  {'DoT Avg':>10}  "
                  f"{'Do53 Err':>9}  {'DoT Err':>9}")
    console.print("-" * 80)

    for dot_pct in SCALING_STEPS:
        do53_target_qps = int(PERF_TOTAL_QPS * (100 - dot_pct) / 100)
        dot_target_qps  = int(PERF_TOTAL_QPS * dot_pct / 100)

        # Build commands
        do53_cmd = [
            "dnsperf", "-s", server,
            "-d", QUERY_INPUT_FILE,
            "-l", str(SCALING_DURATION_SEC),
            "-Q", str(do53_target_qps),
            "-c", str(max(PERF_CONCURRENCY, 2)),
        ]

        dot_total = dot_target_qps * SCALING_DURATION_SEC
        dot_conc = min(max(dot_target_qps // 10, 2), 50)
        dot_cmd = [
            "dnspyre",
            "--server", f"{server}:853",
            "--tls", "--json",
            "-n", str(max(dot_total, 10)),
            "-c", str(dot_conc),
            "--duration", f"{SCALING_DURATION_SEC}s",
        ] + dnspyre_tls_args + query_domains

        # Run both concurrently
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
            f53 = pool.submit(run_cmd, do53_cmd, TOOL_TIMEOUT_SEC)
            fdot = pool.submit(run_cmd, dot_cmd, TOOL_TIMEOUT_SEC)
            out53, err53, rc53 = f53.result()
            outdot, errdot, rcdot = fdot.result()

        p53 = parse_dnsperf(out53 + err53)
        pdot = parse_dnspyre(outdot)

        row = {
            "server": server,
            "dot_pct": dot_pct,
            "do53_qps_target": do53_target_qps,
            "dot_qps_target":  dot_target_qps,
            "do53_qps_actual": p53.get("qps", 0),
            "dot_qps_actual":  pdot.get("qps", 0),
            "do53_latency_avg_ms": p53.get("latency_avg_ms", 0),
            "do53_latency_min_ms": p53.get("latency_min_ms", 0),
            "do53_latency_max_ms": p53.get("latency_max_ms", 0),
            "dot_latency_avg_ms":  pdot.get("latency_avg_ms", 0),
            "dot_latency_p50_ms":  pdot.get("latency_p50_ms", 0),
            "dot_latency_p95_ms":  pdot.get("latency_p95_ms", 0),
            "dot_latency_p99_ms":  pdot.get("latency_p99_ms", 0),
            "do53_errors": int(p53.get("queries_lost", 0) or 0),
            "dot_errors":  int(pdot.get("queries_error", 0) or 0),
        }
        scaling_results.append(row)

        console.print(
            f"{dot_pct:>5}%  "
            f"{row['do53_qps_actual']:>10.1f}  {row['dot_qps_actual']:>10.1f}  "
            f"{row['do53_latency_avg_ms']:>9.2f}ms  {row['dot_latency_avg_ms']:>9.2f}ms  "
            f"{row['do53_errors']:>9}  {row['dot_errors']:>9}"
        )

scaling_df = pd.DataFrame(scaling_results)
console.print("\n[bold green]Scaling test complete.[/bold green]")

In [ ]:
for server in DNS_SERVERS:
    sdf = scaling_df[scaling_df["server"] == server].copy()
    if sdf.empty:
        continue

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # ── Latency vs DoT% ──
    x = sdf["dot_pct"]
    ax1.plot(x, sdf["do53_latency_avg_ms"], "o-", color=COLOR_DO53,
             linewidth=2, label="Do53 avg")
    ax1.plot(x, sdf["dot_latency_avg_ms"], "s-", color=COLOR_DOT,
             linewidth=2, label="DoT avg")
    ax1.plot(x, sdf["dot_latency_p95_ms"], "s--", color=COLOR_DOT,
             linewidth=1, alpha=0.6, label="DoT p95")
    ax1.plot(x, sdf["dot_latency_p99_ms"], "s:", color=COLOR_DOT,
             linewidth=1, alpha=0.4, label="DoT p99")
    ax1.set_xlabel("DoT Traffic (%)")
    ax1.set_ylabel("Latency (ms)")
    ax1.set_title(f"Latency vs DoT% — {server}")
    ax1.legend()
    ax1.xaxis.set_major_locator(mticker.FixedLocator(SCALING_STEPS))

    # ── Throughput & errors vs DoT% ──
    ax2.fill_between(x, 0, sdf["do53_qps_actual"], alpha=0.4,
                     color=COLOR_DO53, label="Do53 QPS")
    ax2.fill_between(x, sdf["do53_qps_actual"],
                     sdf["do53_qps_actual"] + sdf["dot_qps_actual"],
                     alpha=0.4, color=COLOR_DOT, label="DoT QPS")
    ax2.set_xlabel("DoT Traffic (%)")
    ax2.set_ylabel("Queries / sec")
    ax2.set_title(f"Throughput & Errors vs DoT% — {server}")
    ax2.xaxis.set_major_locator(mticker.FixedLocator(SCALING_STEPS))

    ax2_err = ax2.twinx()
    total_err = sdf["do53_errors"] + sdf["dot_errors"]
    ax2_err.plot(x, total_err, "x-", color=COLOR_ERROR, linewidth=1.5,
                 label="Total errors")
    ax2_err.set_ylabel("Errors", color=COLOR_ERROR)
    ax2_err.tick_params(axis="y", labelcolor=COLOR_ERROR)

    # Merge legends
    h1, l1 = ax2.get_legend_handles_labels()
    h2, l2 = ax2_err.get_legend_handles_labels()
    ax2.legend(h1 + h2, l1 + l2, loc="upper left")

    plt.suptitle("Scaling Test Results", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
table = Table(title="Scaling Test Detail", show_lines=True)
table.add_column("Server", style="cyan")
table.add_column("DoT %", justify="right")
table.add_column("Do53 QPS\n(actual)", justify="right")
table.add_column("DoT QPS\n(actual)", justify="right")
table.add_column("Do53 Avg\n(ms)", justify="right")
table.add_column("DoT Avg\n(ms)", justify="right")
table.add_column("DoT p95\n(ms)", justify="right")
table.add_column("Do53\nErrors", justify="right")
table.add_column("DoT\nErrors", justify="right")

for _, r in scaling_df.iterrows():
    table.add_row(
        r["server"],
        f"{int(r['dot_pct'])}%",
        f"{r['do53_qps_actual']:.1f}",
        f"{r['dot_qps_actual']:.1f}",
        f"{r['do53_latency_avg_ms']:.2f}",
        f"{r['dot_latency_avg_ms']:.2f}",
        f"{r['dot_latency_p95_ms']:.2f}",
        str(int(r['do53_errors'])),
        str(int(r['dot_errors'])),
    )

console.print(table)

---
## 9. Executive Summary

In [ ]:
from datetime import datetime

console.print("\n" + "=" * 80)
console.print("[bold]DNS Do53 / DoT Migration Test — Executive Summary[/bold]")
console.print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
console.print(f"Servers tested: {', '.join(DNS_SERVERS)}")
console.print("=" * 80)

# Functional test summary
do53_pass = sum(1 for r in do53_func_results if r["pass"])
do53_total = len(do53_func_results)
dot_pass = sum(1 for r in dot_func_results if r["pass"])
dot_total = len(dot_func_results)
console.print(f"\n[bold]Functional Tests:[/bold]")
console.print(f"  Do53: {do53_pass}/{do53_total} passed")
console.print(f"  DoT:  {dot_pass}/{dot_total} passed")

# Baseline comparison
console.print(f"\n[bold]Performance Baselines ({PERF_DURATION_SEC}s, {PERF_TOTAL_QPS} QPS target):[/bold]")
for server in DNS_SERVERS:
    d53 = next((r for r in do53_perf if r["server"] == server), {})
    dot = next((r for r in dot_perf if r["server"] == server), {})
    console.print(f"  [cyan]{server}[/cyan]:")
    console.print(f"    Do53 — {d53.get('qps',0):.0f} QPS, "
                  f"{d53.get('latency_avg_ms',0):.2f} ms avg latency")
    console.print(f"    DoT  — {dot.get('qps',0):.0f} QPS, "
                  f"{dot.get('latency_avg_ms',0):.2f} ms avg latency, "
                  f"{dot.get('latency_p95_ms',0):.2f} ms p95")
    avg53 = d53.get("latency_avg_ms", 0)
    avgdot = dot.get("latency_avg_ms", 0)
    if avg53 > 0:
        overhead = ((avgdot - avg53) / avg53) * 100
        console.print(f"    [bold]DoT overhead: {overhead:+.1f}%[/bold]")

# Scaling summary
if not scaling_df.empty:
    console.print(f"\n[bold]Scaling Test ({PERF_TOTAL_QPS} total QPS, "
                  f"{SCALING_DURATION_SEC}s per step):[/bold]")
    for server in DNS_SERVERS:
        sdf = scaling_df[scaling_df["server"] == server]
        if sdf.empty:
            continue
        console.print(f"  [cyan]{server}[/cyan]:")
        # Find the step where DoT p95 first exceeds a threshold
        max_dot_p95 = sdf["dot_latency_p95_ms"].max()
        min_dot_p95 = sdf["dot_latency_p95_ms"].min()
        console.print(f"    DoT p95 range: {min_dot_p95:.2f} ms — {max_dot_p95:.2f} ms")
        total_errors = sdf["do53_errors"].sum() + sdf["dot_errors"].sum()
        console.print(f"    Total errors across all steps: {total_errors}")
        # Check if latency degrades significantly
        first_avg = sdf.iloc[0]["dot_latency_avg_ms"]
        last_avg = sdf.iloc[-1]["dot_latency_avg_ms"]
        if first_avg > 0:
            change = ((last_avg - first_avg) / first_avg) * 100
            console.print(f"    DoT avg latency change (10%→90%): {change:+.1f}%")

console.print("\n" + "=" * 80)

---
## 10. Export

Run the cell below to export this notebook (with all outputs) to PDF.

In [ ]:
import os
nb_path = os.path.abspath("dns-dot-migration-test.ipynb")
if not os.path.exists(nb_path):
    # Try the data directory path
    nb_path = "/workspace/notebooks/dns-dot-migration-test.ipynb"
out_dir = os.path.dirname(nb_path)
!jupyter nbconvert --to pdf "{nb_path}" --output-dir "{out_dir}" 2>&1 | tail -5
print(f"\nPDF saved to: {out_dir}/dns-dot-migration-test.pdf")